# NEXT_STEPS Comprehensive Walkthrough

This notebook fully demonstrates every research direction from `NEXT_STEPS.md`:

| Track | Day | Direction |
|-------|-----|-----------|
| A | 1–2 | Student-t ν sweep `{3, 5, 8, learned}` + post-hoc variance scaling |
| B | 3   | Fixed-variance mentor direction (global constant + epoch-scheduled) |
| C | 4   | Ranking-aware auxiliary loss (pairwise hinge) |
| D | 5   | Uncertainty-feature ablation (RSA proxy + mutation-type context) |

**Synthetic data** is used so the notebook runs without real ESM2 embeddings, but
the data generator mirrors the heteroscedastic structure expected from T2837 mutations:
different mutation types carry different intrinsic noise levels, and RSA (solvent
exposure) modulates the variance.

**Success criteria** (from NEXT_STEPS):
- ICE ≤ 0.02
- NLL < 1.85 without RMSE degradation > 1%
- Spearman(σ, |error|) significantly > 0

The final cell assembles the complete **deliverable table**:
RMSE · MAE · NLL · ICE · coverage@{50,80,90,95}% · bias-by-group · Spearman · top-k risk capture.

## 0  Setup

In [ ]:
import sys, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import spearmanr
from scipy.optimize import minimize

# Make sure the uapp package is on the path regardless of CWD
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from uapp.heads import build_head, TwoHeadNLL
from uapp.losses import (
    mse_loss, gaussian_nll_loss, student_t_nll_loss,
    uncertainty_ranking_loss,
)
from uapp.train import TrainConfig, train_head
from uapp.evaluate import (
    EvalResult, evaluate_head,
    compute_rmse, compute_mae, compute_gaussian_nll,
    compute_ice, compute_coverage_curve,
    compute_spearman_sigma_error, compute_top_k_risk_capture,
    plot_reliability_diagram,
)
from uapp.data import make_loader
from uapp.utils import set_seed, get_device

warnings.filterwarnings('ignore')

DEVICE = get_device(prefer_cuda=False)  # use CPU for notebook reproducibility
SEED   = 42
set_seed(SEED)
print(f"Device: {DEVICE}")

## 1  Synthetic Data Generator

The generator produces a dataset that mimics what we expect from T2837 ESM2 embeddings:

- **Embeddings** `h` ∈ ℝ^d are Gaussian with a few dominant principal directions.
- **True signal** is a linear function of the first few PCs with added nonlinearity.
- **Noise variance** depends on `RSA` (solvent exposure proxy) and `mut_type`
  (4 classes: polar↔nonpolar, charge-flip, synonymous), mimicking the heteroscedastic
  structure highlighted in the NEXT_STEPS memo.

This lets us assess whether each head correctly identifies high-variance regions.

In [ ]:
def make_synthetic_dataset(
    n: int = 2000,
    d: int = 320,
    n_signal_dims: int = 10,
    seed: int = 42,
) -> dict:
    """
    Returns dict with keys:
      X        : (n, d) embedding tensor
      y        : (n,)   ddG label tensor
      rsa      : (n,)   RSA proxy in [0,1]
      mut_type : (n,)   int in {0,1,2,3}
      true_sigma: (n,)  ground-truth noise sigma (oracle)
    """
    rng = np.random.default_rng(seed)

    # Embeddings: structured covariance
    factors = rng.standard_normal((n, n_signal_dims)).astype(np.float32)
    loadings = rng.standard_normal((n_signal_dims, d)).astype(np.float32) * 0.3
    noise_emb = rng.standard_normal((n, d)).astype(np.float32) * 0.05
    X = factors @ loadings + noise_emb

    # Mutation type and RSA proxy (independent of X to keep identification clean)
    mut_type = rng.integers(0, 4, size=n)          # 0=synonymous,1=polar,2=nonpolar,3=charge
    rsa = rng.beta(2.0, 3.0, size=n).astype(np.float32)  # skewed toward buried

    # True mean: nonlinear function of first 5 PCs
    beta = rng.standard_normal(n_signal_dims).astype(np.float32)
    mu_true = factors @ beta
    mu_true += 0.3 * (factors[:, 0] * factors[:, 1])  # mild interaction

    # Heteroscedastic noise: sigma depends on RSA and mut_type
    sigma_base = np.array([0.5, 1.0, 1.5, 2.5], dtype=np.float32)  # per mut_type
    true_sigma = sigma_base[mut_type] * (0.5 + rsa)  # higher RSA → more noise

    y = mu_true + rng.standard_normal(n).astype(np.float32) * true_sigma

    return dict(
        X=torch.from_numpy(X),
        y=torch.from_numpy(y),
        rsa=torch.from_numpy(rsa),
        mut_type=torch.from_numpy(mut_type.astype(np.int64)),
        true_sigma=torch.from_numpy(true_sigma),
    )


def split_dataset(ds: dict, train_frac=0.7, val_frac=0.15, seed=42):
    n = ds['X'].shape[0]
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)
    n_tr = int(n * train_frac)
    n_va = int(n * val_frac)
    splits = {}
    for name, sl in [("train", idx[:n_tr]), ("val", idx[n_tr:n_tr+n_va]), ("test", idx[n_tr+n_va:])]:
        splits[name] = {k: v[sl] for k, v in ds.items()}
    return splits


# ── Generate ─────────────────────────────────────────────────────────────────
FULL = make_synthetic_dataset(n=2000, d=320, seed=SEED)
SPLITS = split_dataset(FULL, seed=SEED)

D_IN = FULL['X'].shape[1]

# DataLoaders (X, y only — structural features handled in Track D)
LOADERS = {
    split: make_loader(s['X'], s['y'], batch_size=128, shuffle=(split == 'train'))
    for split, s in SPLITS.items()
}

print(f"d={D_IN}  |  n_train={len(SPLITS['train']['y'])}  "
      f"n_val={len(SPLITS['val']['y'])}  n_test={len(SPLITS['test']['y'])}")
print(f"y: mean={FULL['y'].mean():.3f}  std={FULL['y'].std():.3f}")
print(f"Oracle sigma: mean={FULL['true_sigma'].mean():.3f}  "
      f"range=[{FULL['true_sigma'].min():.2f}, {FULL['true_sigma'].max():.2f}]")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

axes[0].hist(FULL['y'].numpy(), bins=40, color='#2C4A6E', alpha=0.8)
axes[0].set_title('ddG distribution')
axes[0].set_xlabel('ddG (kcal/mol)')

for mt, label in enumerate(['synonymous', 'polar', 'nonpolar', 'charge-flip']):
    mask = FULL['mut_type'] == mt
    axes[1].scatter(
        FULL['rsa'][mask].numpy(),
        FULL['true_sigma'][mask].numpy(),
        s=4, alpha=0.4, label=label
    )
axes[1].set_title('Oracle σ vs RSA (by mut type)')
axes[1].set_xlabel('RSA')
axes[1].set_ylabel('True σ')
axes[1].legend(fontsize=7, markerscale=3)

axes[2].scatter(FULL['y'].numpy(), FULL['true_sigma'].numpy(), s=4, alpha=0.3, color='#9B5FA7')
axes[2].set_title('Oracle σ vs ddG')
axes[2].set_xlabel('ddG')
axes[2].set_ylabel('True σ')

plt.tight_layout()
plt.savefig('figures/synth_dataset.png', dpi=120)
plt.show()
print("Saved figures/synth_dataset.png")

## 2  Shared Helpers

Helper functions used across all tracks: quick training wrapper, prediction
extraction, post-hoc variance scaling, and a unified metrics dict builder.

In [ ]:
# ── Quick training wrapper ────────────────────────────────────────────────────
def quick_train(
    head: nn.Module,
    cfg: TrainConfig,
    loaders: dict = LOADERS,
    device: torch.device = DEVICE,
) -> nn.Module:
    head, _ = train_head(head, loaders['train'], loaders['val'], cfg, device)
    return head


# ── Prediction extraction ─────────────────────────────────────────────────────
def predict(head: nn.Module, loader, device=DEVICE):
    """Return (mu, sigma, y) numpy arrays. sigma=None for deterministic heads."""
    from uapp.heads import is_probabilistic
    head.eval().to(device)
    mus, sigs, ys = [], [], []
    prob = is_probabilistic(head)
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            if prob:
                mu, sigma = head(xb)
                sigs.append(sigma.cpu().numpy())
            else:
                mu = head(xb)
            mus.append(mu.cpu().numpy())
            ys.append(yb.numpy())
    mu_arr = np.concatenate(mus)
    y_arr  = np.concatenate(ys)
    sig_arr = np.concatenate(sigs) if sigs else None
    return mu_arr, sig_arr, y_arr


# ── Post-hoc variance scaling ─────────────────────────────────────────────────
def fit_scale_one(mu, sigma, y):
    """Fit sigma' = a * sigma by minimising val NLL. Returns scalar a."""
    def obj(x):
        a = float(x[0])
        s = np.maximum(a * sigma, 1e-6)
        var = s**2
        return float(np.mean(0.5 * ((y - mu)**2 / var + np.log(2*math.pi*var))))
    res = minimize(obj, x0=[1.0], bounds=[(1e-4, 10.0)])
    return float(res.x[0]) if res.success else 1.0


def fit_scale_two(mu, sigma, y):
    """Fit sigma' = a * sigma + b by minimising val NLL. Returns (a, b)."""
    def obj(x):
        a, b = float(x[0]), float(x[1])
        s = np.maximum(a * sigma + b, 1e-6)
        var = s**2
        return float(np.mean(0.5 * ((y - mu)**2 / var + np.log(2*math.pi*var))))
    res = minimize(obj, x0=[1.0, 0.0], bounds=[(1e-4, 10.0), (0.0, 5.0)])
    return (float(res.x[0]), float(res.x[1])) if res.success else (1.0, 0.0)


def apply_scaling(mu, sigma, a, b=0.0):
    return np.maximum(a * sigma + b, 1e-6)


# ── Metrics dict builder ──────────────────────────────────────────────────────
def metrics_dict(name: str, mu, sigma, y) -> dict:
    """Compute all deliverable-table metrics and return a flat dict."""
    d = {"model": name}
    d["rmse"] = compute_rmse(mu, y)
    d["mae"]  = compute_mae(mu, y)
    if sigma is not None:
        d["nll"]  = compute_gaussian_nll(mu, sigma, y)
        ice, cov  = compute_ice(mu, sigma, y)
        d["ice"]  = ice
        for k, v in cov.items():
            d[f"cov@{k}"] = v
        d["spearman"] = compute_spearman_sigma_error(mu, sigma, y)
        for k, v in compute_top_k_risk_capture(mu, sigma, y).items():
            d[f"top{k}"] = v
    return d


# Collect all results here
ALL_RESULTS: list[dict] = []

print("Helpers loaded.")

---
## Track A  —  Student-t ν Sweep + Post-hoc Variance Scaling

**NEXT_STEPS Day 1–2**

Train `TwoHeadNLL` with the Student-t likelihood for ν ∈ {3, 5, 8, **learned**}.
For each model, fit two post-hoc scalings on the *validation* set only:

- One-parameter: `σ' = a·σ`
- Two-parameter: `σ' = a·σ + b`

Evaluate on the held-out **test** split.

In [ ]:
BASE_CFG = dict(
    max_epochs=200,
    lr=1e-3,
    weight_decay=1e-5,
    patience=25,
    log_every=999,  # suppress per-epoch logs in notebook
)

track_a_heads = {}   # name → trained head
track_a_nu_vals = {} # name → learned nu (for learned variant)

for nu_setting in [3.0, 5.0, 8.0, "learned"]:
    learn_nu = (nu_setting == "learned")
    label    = f"StudentT_nu={nu_setting}"
    print(f"  Training {label} ...", end=" ", flush=True)

    head = build_head(
        "two_head_nll", D_IN,
        d_hidden=128, dropout=0.1,
        init_sigma_bias=0.5,
        learn_nu=learn_nu,
        init_nu=3.0 if learn_nu else float(nu_setting),
    )

    cfg = TrainConfig(
        **BASE_CFG,
        loss_type="student_t",
        student_t_nu=float(nu_setting) if not learn_nu else 3.0,  # ignored when learn_nu
    )
    head = quick_train(head, cfg)
    track_a_heads[label] = head

    if learn_nu:
        nu_learned = float(head.nu.item())
        track_a_nu_vals[label] = nu_learned
        print(f"done  (learned ν = {nu_learned:.3f})")
    else:
        print("done")

print("\nAll Track A models trained.")

In [ ]:
track_a_rows = []

# Val predictions (for fitting scalers)
val_preds = {
    name: predict(head, LOADERS['val'])
    for name, head in track_a_heads.items()
}
# Test predictions
te_preds = {
    name: predict(head, LOADERS['test'])
    for name, head in track_a_heads.items()
}

for name in track_a_heads:
    mu_va, sig_va, y_va = val_preds[name]
    mu_te, sig_te, y_te = te_preds[name]

    # Raw (no scaling)
    row = metrics_dict(f"{name}  [raw]", mu_te, sig_te, y_te)
    row["track"] = "A"; row["scaled"] = "none"
    track_a_rows.append(row); ALL_RESULTS.append(row)

    # One-parameter scaling
    a1 = fit_scale_one(mu_va, sig_va, y_va)
    sig_1 = apply_scaling(mu_te, sig_te, a1)
    row1 = metrics_dict(f"{name}  [scale a]", mu_te, sig_1, y_te)
    row1["track"] = "A"; row1["scaled"] = f"a={a1:.3f}"
    track_a_rows.append(row1); ALL_RESULTS.append(row1)

    # Two-parameter scaling
    a2, b2 = fit_scale_two(mu_va, sig_va, y_va)
    sig_2 = apply_scaling(mu_te, sig_te, a2, b2)
    row2 = metrics_dict(f"{name}  [scale a+b]", mu_te, sig_2, y_te)
    row2["track"] = "A"; row2["scaled"] = f"a={a2:.3f},b={b2:.3f}"
    track_a_rows.append(row2); ALL_RESULTS.append(row2)

df_a = pd.DataFrame(track_a_rows)
cols_show = ["model", "rmse", "nll", "ice", "spearman", "top0.10", "top0.20"]
print(df_a[[c for c in cols_show if c in df_a.columns]]
      .sort_values("ice")
      .to_string(index=False, float_format="{:.4f}".format))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: reliability diagrams (raw models)
ax = axes[0]
ax.plot([0, 1], [0, 1], '--', color='#888', lw=0.8, label='perfect')
colors = ['#0D7377', '#E8913A', '#2C4A6E', '#9B5FA7']
for i, (name, head) in enumerate(track_a_heads.items()):
    mu, sig, y = te_preds[name]
    nom, emp = compute_coverage_curve(mu, sig, y)
    ax.plot(nom, emp, 'o-', ms=4, lw=1.5, color=colors[i],
            label=name.replace('StudentT_', ''))
ax.set_xlabel('Nominal coverage'); ax.set_ylabel('Empirical coverage')
ax.set_title('Track A – reliability diagrams (raw)'); ax.legend(fontsize=8)
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_aspect('equal'); ax.grid(alpha=0.3)

# Right: NLL vs ICE scatter (all variants including scaled)
ax2 = axes[1]
styles = {'none': 'o', 'a': 's', 'a+b': '^'}
for _, row in df_a.iterrows():
    if 'nll' not in row or pd.isna(row.get('nll')): continue
    marker = 's' if 'scale a+b' in row['model'] else ('s' if 'scale a]' in row['model'] else 'o')
    ax2.scatter(row['nll'], row['ice'], s=60, marker=marker, alpha=0.8)
    ax2.annotate(
        row['model'].replace('StudentT_nu=', 'ν=').split('[')[0].strip(),
        (row['nll'], row['ice']), fontsize=7, ha='center', va='bottom'
    )
ax2.set_xlabel('NLL'); ax2.set_ylabel('ICE')
ax2.set_title('Track A – NLL vs ICE (all scaling variants)'); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/track_a_results.png', dpi=120)
plt.show()
print("Saved figures/track_a_results.png")

In [ ]:
# Inspect learned ν values
print("Learned ν values (Track A):")
for name, nu in track_a_nu_vals.items():
    print(f"  {name}: ν = {nu:.4f}")

# Variance scaling diagnostics: compare a-only vs a+b on val NLL
print("\nPost-hoc scaling diagnostics on val set:")
for name in track_a_heads:
    mu_va, sig_va, y_va = val_preds[name]
    a1 = fit_scale_one(mu_va, sig_va, y_va)
    a2, b2 = fit_scale_two(mu_va, sig_va, y_va)
    nll_raw = compute_gaussian_nll(mu_va, sig_va, y_va)
    nll_1   = compute_gaussian_nll(mu_va, apply_scaling(mu_va, sig_va, a1), y_va)
    nll_2   = compute_gaussian_nll(mu_va, apply_scaling(mu_va, sig_va, a2, b2), y_va)
    label   = name.replace('StudentT_', '')
    print(f"  {label:20s}  raw NLL={nll_raw:.4f}  a-only={nll_1:.4f}(a={a1:.3f})  a+b={nll_2:.4f}(a={a2:.3f},b={b2:.3f})")

---
## Track B  —  Fixed Variance + Mean Calibration (Mentor Direction)

**NEXT_STEPS Day 3**

Test whether calibration gains come from variance flexibility or from better
point estimates by constraining σ and forcing the model to improve μ.

- **Variant A**: global `σ = c` tuned on validation NLL (`c ∈ {1.0, 1.5, 2.0}`).
- **Variant B**: epoch-scheduled `σ_t` — starts large and decays, nudging the
  mean pathway to fit tighter over time.

The `FixedSigmaNLL` head is used for both; Variant B wraps training in a scheduler.

In [ ]:
# ── Variant A: global fixed sigma ─────────────────────────────────────────────
track_b_heads_A = {}

for sigma_c in [1.0, 1.5, 2.0]:
    label = f"FixedSigma_A={sigma_c:.1f}"
    print(f"  Training {label} ...", end=" ", flush=True)
    head = build_head(
        "fixed_sigma_nll", D_IN,
        d_hidden=128, dropout=0.1, fixed_sigma=sigma_c,
    )
    cfg = TrainConfig(**BASE_CFG, loss_type="gaussian")
    head = quick_train(head, cfg)
    track_b_heads_A[label] = head
    print("done")

# Grid-search optimal fixed sigma on validation NLL
print("\nGrid-searching optimal fixed σ on val NLL:")
sigma_grid = np.linspace(0.5, 3.0, 26)
best_sigma_per_model = {}

for name, head in track_b_heads_A.items():
    mu_va, _, y_va = predict(head, LOADERS['val'])
    best_nll = np.inf
    best_s   = None
    for s in sigma_grid:
        sig_const = np.full_like(mu_va, s)
        nll = compute_gaussian_nll(mu_va, sig_const, y_va)
        if nll < best_nll:
            best_nll = nll; best_s = s
    best_sigma_per_model[name] = best_s
    print(f"  {name:30s}  best σ={best_s:.3f}  val NLL={best_nll:.4f}")

In [ ]:
# ── Variant B: epoch-scheduled sigma ─────────────────────────────────────────
# We implement a simple sigma schedule by training with two-stage TrainConfig:
# Stage 1: high fixed sigma (σ=2.5) – model learns a rough mean
# Stage 2: low fixed sigma (σ=1.0) – model refines mean under tighter constraint

def train_scheduled_sigma(
    d_in: int, sigma_start=2.5, sigma_end=1.0,
    n_stages=3, epochs_per_stage=60, loaders=LOADERS, device=DEVICE,
) -> nn.Module:
    """Train with a step-wise decreasing fixed sigma."""
    sigma_schedule = np.linspace(sigma_start, sigma_end, n_stages)
    head = build_head(
        "fixed_sigma_nll", d_in,
        d_hidden=128, dropout=0.1, fixed_sigma=float(sigma_schedule[0]),
    )
    for stage, sigma in enumerate(sigma_schedule):
        # Update the head's buffer for this stage
        head.fixed_sigma.fill_(float(sigma))
        cfg = TrainConfig(
            max_epochs=epochs_per_stage, lr=1e-3,
            weight_decay=1e-5, patience=15, log_every=999,
            loss_type="gaussian",
        )
        head, _ = train_head(head, loaders['train'], loaders['val'], cfg, device)
        print(f"    Stage {stage+1}/{n_stages}  σ={sigma:.2f} done")
    return head


print("Training FixedSigma Variant B (scheduled sigma: 2.5 → 1.0) ...")
head_b_scheduled = train_scheduled_sigma(D_IN)
track_b_heads_A["FixedSigma_B_scheduled"] = head_b_scheduled
print("Done.")

In [ ]:
track_b_rows = []

for name, head in track_b_heads_A.items():
    mu_te, sig_te_raw, y_te = predict(head, LOADERS['test'])

    # For fixed-sigma heads, inject the tuned global sigma
    if name in best_sigma_per_model:
        best_s = best_sigma_per_model[name]
        sig_te = np.full_like(mu_te, best_s)
    else:
        # Scheduled head: use head's current sigma buffer
        sig_te = np.full_like(mu_te, float(head.fixed_sigma.item()))
        # Also grid-search best sigma on val
        mu_va, _, y_va = predict(head, LOADERS['val'])
        best_s_b = min(
            np.linspace(0.5, 3.0, 26),
            key=lambda s: compute_gaussian_nll(mu_va, np.full_like(mu_va, s), y_va),
        )
        sig_te = np.full_like(mu_te, float(best_s_b))

    row = metrics_dict(name, mu_te, sig_te, y_te)
    row["track"] = "B"
    track_b_rows.append(row)
    ALL_RESULTS.append(row)

df_b = pd.DataFrame(track_b_rows)
cols_show = ["model", "rmse", "mae", "nll", "ice", "spearman"]
print("Track B results:")
print(df_b[[c for c in cols_show if c in df_b.columns]]
      .sort_values("rmse")
      .to_string(index=False, float_format="{:.4f}".format))

In [ ]:
# Bias-by-mutation-group analysis (mentor direction: penalize bias across groups)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
mut_labels = ['synonymous', 'polar↔nonpolar', 'nonpolar', 'charge-flip']
mut_type_te = SPLITS['test']['mut_type'].numpy()

ax = axes[0]
for name, head in track_b_heads_A.items():
    mu_te, _, y_te = predict(head, LOADERS['test'])
    bias_per_group = [
        float(np.mean(mu_te[mut_type_te == mt] - y_te[mut_type_te == mt]))
        for mt in range(4)
    ]
    ax.plot(mut_labels, bias_per_group, 'o-', ms=5, lw=1.2, label=name.replace('FixedSigma_', ''))

ax.axhline(0, color='k', lw=0.8, linestyle='--')
ax.set_ylabel('Mean bias (pred − true)')
ax.set_title('Track B – bias by mutation type')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot([0,1],[0,1],'--', color='#888', lw=0.8, label='perfect')
for i, (name, head) in enumerate(track_b_heads_A.items()):
    mu_te, _, y_te = predict(head, LOADERS['test'])
    if name in best_sigma_per_model:
        sig = np.full_like(mu_te, best_sigma_per_model[name])
    else:
        mu_va, _, y_va = predict(head, LOADERS['val'])
        best_s = min(np.linspace(0.5,3.0,26),
                     key=lambda s: compute_gaussian_nll(mu_va, np.full_like(mu_va, s), y_va))
        sig = np.full_like(mu_te, float(best_s))
    nom, emp = compute_coverage_curve(mu_te, sig, y_te)
    ax2.plot(nom, emp, 'o-', ms=4, lw=1.5, label=name.replace('FixedSigma_',''), color=colors[i % 4])
ax2.set_xlim(0,1); ax2.set_ylim(0,1); ax2.set_aspect('equal')
ax2.set_xlabel('Nominal'); ax2.set_ylabel('Empirical')
ax2.set_title('Track B – reliability diagrams'); ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/track_b_results.png', dpi=120)
plt.show()
print("Saved figures/track_b_results.png")

---
## Track C  —  Ranking-Aware Uncertainty

**NEXT_STEPS Day 4**

Address the near-zero Spearman(σ, |error|) by adding a pairwise hinge loss that
encourages monotonicity between predicted σ and absolute residuals:

```
L_rank = mean_{i,j: |e_i|>|e_j|} max(0, margin − (σ_i − σ_j))
```

Combined objective: `L = L_StudentT(ν=3) + λ · L_rank`

We sweep `λ ∈ {0, 0.005, 0.01, 0.05}` and report:
- Spearman(σ, |error|)
- top-k risk capture at k ∈ {10%, 20%, 30%}
- AURC (Area Under the Risk-Coverage curve)
- RMSE / NLL / ICE (to confirm no regression)

In [ ]:
track_c_heads = {}

for lam in [0.0, 0.005, 0.01, 0.05]:
    label = f"Rank_lambda={lam}"
    print(f"  Training {label} ...", end=" ", flush=True)
    head = build_head(
        "two_head_nll", D_IN,
        d_hidden=128, dropout=0.1, init_sigma_bias=0.5,
    )
    cfg = TrainConfig(
        **BASE_CFG,
        loss_type="student_t", student_t_nu=3.0,
        ranking_lambda=lam, ranking_margin=0.05,
    )
    head = quick_train(head, cfg)
    track_c_heads[label] = head
    print("done")

print("\nAll Track C models trained.")

In [ ]:
def compute_aurc(mu: np.ndarray, sigma: np.ndarray, y: np.ndarray) -> float:
    """Area Under the Risk-Coverage curve (lower is better).

    Sort predictions by ascending sigma (most confident first).
    At each threshold, report MSE on the retained subset.
    Integrate over thresholds.
    """
    n = len(mu)
    order = np.argsort(sigma)  # most confident first
    abs_err2 = (mu - y) ** 2
    # Compute running mean MSE as we include more samples
    cumsum = np.cumsum(abs_err2[order])
    k_arr  = np.arange(1, n + 1)
    risks  = cumsum / k_arr
    # Integrate (trapezoidal) over coverage in [0,1]
    coverage = k_arr / n
    return float(np.trapz(risks, coverage))


track_c_rows = []
for name, head in track_c_heads.items():
    mu_te, sig_te, y_te = predict(head, LOADERS['test'])
    row = metrics_dict(name, mu_te, sig_te, y_te)
    row["aurc"]  = compute_aurc(mu_te, sig_te, y_te)
    row["track"] = "C"
    track_c_rows.append(row)
    ALL_RESULTS.append(row)

df_c = pd.DataFrame(track_c_rows)
cols_show = ["model", "rmse", "nll", "ice", "spearman", "top0.10", "top0.20", "aurc"]
print("Track C results:")
print(df_c[[c for c in cols_show if c in df_c.columns]]
      .to_string(index=False, float_format="{:.4f}".format))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Left: Spearman vs lambda
lambdas = [0.0, 0.005, 0.01, 0.05]
spearmans = [r['spearman'] for r in track_c_rows]
axes[0].plot(lambdas, spearmans, 'o-', color='#0D7377', lw=2, ms=7)
axes[0].axhline(0, color='k', lw=0.8, linestyle='--')
axes[0].set_xlabel('Ranking λ'); axes[0].set_ylabel('Spearman(σ, |error|)')
axes[0].set_title('Track C – ranking λ vs Spearman'); axes[0].grid(alpha=0.3)

# Middle: AURC curves
ax = axes[1]
for i, (name, head) in enumerate(track_c_heads.items()):
    mu_te, sig_te, y_te = predict(head, LOADERS['test'])
    n = len(mu_te)
    order = np.argsort(sig_te)
    abs_err2 = (mu_te - y_te)**2
    risks = np.cumsum(abs_err2[order]) / np.arange(1, n+1)
    coverage = np.arange(1, n+1) / n
    lam_val = lambdas[i]
    ax.plot(coverage, risks, lw=1.5, label=f'λ={lam_val}', color=colors[i % 4])
ax.set_xlabel('Coverage (fraction retained)')
ax.set_ylabel('Risk (MSE on retained)')
ax.set_title('Track C – risk-coverage curves'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Right: σ vs |error| scatter for λ=0 vs best λ
ax3 = axes[2]
best_lam = lambdas[int(np.argmax(spearmans))]
for lam_val, marker, c in [(0.0, 'o', '#888888'), (best_lam, '^', '#E8913A')]:
    head = track_c_heads[f"Rank_lambda={lam_val}"]
    mu, sig, y = predict(head, LOADERS['test'])
    idx = np.random.choice(len(mu), 300, replace=False)
    ax3.scatter(sig[idx], np.abs(y-mu)[idx], s=12, alpha=0.4,
                marker=marker, color=c, label=f'λ={lam_val}')
ax3.set_xlabel('Predicted σ'); ax3.set_ylabel('|error|')
ax3.set_title('Track C – σ vs |error| (λ=0 vs best λ)'); ax3.legend(); ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/track_c_results.png', dpi=120)
plt.show()
print("Saved figures/track_c_results.png")

---
## Track D  —  Structure-Aware Uncertainty Features

**NEXT_STEPS Day 5**

Test whether RSA (solvent exposure) and mutation-type features improve the
**variance branch** without touching the mean branch.

Architecture: a `FeatureAugmentedHead` where the σ-MLP takes concatenated
`[h_G, rsa, mut_type_onehot]` while the μ-MLP takes only `h_G`.

Ablation:
1. Baseline: no structural features (standard `TwoHeadNLL`)
2. RSA only
3. Mutation type only
4. RSA + mutation type (full)

In [ ]:
import torch.nn.functional as F


def _mlp(d_in, d_hidden, d_out, dropout=0.1):
    return nn.Sequential(
        nn.Linear(d_in, d_hidden), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(d_hidden, d_out),
    )


class FeatureAugmentedHead(nn.Module):
    """Mean branch: h_G only.  Variance branch: h_G + optional [rsa, mut_type].

    Extra features are passed as a second argument to forward().
    If None, falls back to h_G only for sigma (identical to TwoHeadNLL).
    """
    N_MUT_TYPES = 4

    def __init__(
        self, d_in, d_hidden=128, dropout=0.1,
        use_rsa=False, use_mut_type=False,
        sigma_floor=1e-6, init_sigma_bias=0.5,
    ):
        super().__init__()
        self.use_rsa      = use_rsa
        self.use_mut_type = use_mut_type
        self.sigma_floor  = sigma_floor

        d_extra = int(use_rsa) + (self.N_MUT_TYPES if use_mut_type else 0)
        self.mu_mlp    = _mlp(d_in, d_hidden, 1, dropout)
        self.sigma_mlp = _mlp(d_in + d_extra, d_hidden, 1, dropout)

        with torch.no_grad():
            self.sigma_mlp[-1].bias.fill_(init_sigma_bias)

    def forward(
        self, h: torch.Tensor,
        rsa: torch.Tensor | None = None,
        mut_type: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        mu  = self.mu_mlp(h).squeeze(-1)

        parts = [h]
        if self.use_rsa and rsa is not None:
            parts.append(rsa.unsqueeze(-1).float())
        if self.use_mut_type and mut_type is not None:
            ohe = F.one_hot(mut_type, self.N_MUT_TYPES).float()
            parts.append(ohe)
        h_sigma = torch.cat(parts, dim=-1)

        raw   = self.sigma_mlp(h_sigma).squeeze(-1)
        sigma = F.softplus(raw) + self.sigma_floor
        return mu, sigma


# Feature-aware DataLoader wrapper
from torch.utils.data import TensorDataset, DataLoader

def make_feature_loader(split_data, batch_size=128, shuffle=False):
    ds = TensorDataset(
        split_data['X'].float(),
        split_data['y'].float(),
        split_data['rsa'],
        split_data['mut_type'],
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

FEAT_LOADERS = {
    split: make_feature_loader(s, shuffle=(split=='train'))
    for split, s in SPLITS.items()
}

print("FeatureAugmentedHead defined.")

In [ ]:
def train_feature_head(
    head: FeatureAugmentedHead,
    loaders: dict,
    max_epochs=200, lr=1e-3, weight_decay=1e-5, patience=25,
    nu=3.0, device=DEVICE,
) -> FeatureAugmentedHead:
    """Custom training loop for FeatureAugmentedHead (takes rsa, mut_type)."""
    head = head.to(device)
    opt = torch.optim.Adam(head.parameters(), lr=lr, weight_decay=weight_decay)
    best_val, best_state, patience_ctr = np.inf, None, 0

    for epoch in range(1, max_epochs + 1):
        # Train
        head.train()
        for xb, yb, rsa_b, mt_b in loaders['train']:
            xb, yb = xb.to(device), yb.to(device)
            rsa_b, mt_b = rsa_b.to(device), mt_b.to(device)
            opt.zero_grad()
            mu, sig = head(xb, rsa_b, mt_b)
            loss = student_t_nll_loss(mu, sig, yb, nu=nu)
            loss.backward(); opt.step()
        # Val
        head.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb, rsa_b, mt_b in loaders['val']:
                xb, yb = xb.to(device), yb.to(device)
                rsa_b, mt_b = rsa_b.to(device), mt_b.to(device)
                mu, sig = head(xb, rsa_b, mt_b)
                val_losses.append(float(student_t_nll_loss(mu, sig, yb, nu=nu).item()))
        val_loss = np.mean(val_losses)
        if val_loss < best_val - 1e-4:
            best_val = val_loss
            import copy; best_state = copy.deepcopy(head.state_dict())
            patience_ctr = 0
        else:
            patience_ctr += 1
        if patience_ctr >= patience:
            break

    if best_state:
        head.load_state_dict(best_state)
    return head


def predict_feature_head(head, loader, device=DEVICE):
    head.eval().to(device)
    mus, sigs, ys = [], [], []
    with torch.no_grad():
        for xb, yb, rsa_b, mt_b in loader:
            xb, rsa_b, mt_b = xb.to(device), rsa_b.to(device), mt_b.to(device)
            mu, sig = head(xb, rsa_b, mt_b)
            mus.append(mu.cpu().numpy())
            sigs.append(sig.cpu().numpy())
            ys.append(yb.numpy())
    return np.concatenate(mus), np.concatenate(sigs), np.concatenate(ys)


# ── Ablation: 4 variants ─────────────────────────────────────────────────────
track_d_variants = {
    "D_baseline":     dict(use_rsa=False, use_mut_type=False),
    "D_RSA_only":     dict(use_rsa=True,  use_mut_type=False),
    "D_MutType_only": dict(use_rsa=False, use_mut_type=True),
    "D_RSA+MutType":  dict(use_rsa=True,  use_mut_type=True),
}

track_d_heads = {}
for name, kwargs in track_d_variants.items():
    print(f"  Training {name} ...", end=" ", flush=True)
    head = FeatureAugmentedHead(D_IN, **kwargs)
    head = train_feature_head(head, FEAT_LOADERS)
    track_d_heads[name] = head
    print("done")

print("\nAll Track D models trained.")

In [ ]:
track_d_rows = []
for name, head in track_d_heads.items():
    mu_te, sig_te, y_te = predict_feature_head(head, FEAT_LOADERS['test'])
    row = metrics_dict(name, mu_te, sig_te, y_te)
    row["track"] = "D"
    track_d_rows.append(row)
    ALL_RESULTS.append(row)

df_d = pd.DataFrame(track_d_rows)
cols_show = ["model", "rmse", "nll", "ice", "spearman", "top0.10", "top0.20", "top0.30"]
print("Track D results (ablation):")
print(df_d[[c for c in cols_show if c in df_d.columns]]
      .to_string(index=False, float_format="{:.4f}".format))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: bar chart of ICE and Spearman per ablation variant
variant_names = list(track_d_heads.keys())
ices     = [r['ice'] for r in track_d_rows]
spearmans_d = [r['spearman'] for r in track_d_rows]

x = np.arange(len(variant_names))
w = 0.35
axes[0].bar(x - w/2, ices,      w, label='ICE',     color='#0D7377', alpha=0.8)
axes[0].bar(x + w/2, spearmans_d, w, label='Spearman', color='#E8913A', alpha=0.8)
axes[0].axhline(0, color='k', lw=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(
    [n.replace('D_', '') for n in variant_names], rotation=15, ha='right'
)
axes[0].set_title('Track D – ICE and Spearman by feature ablation')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

# Right: σ vs oracle σ for best variant
best_d = min(track_d_heads, key=lambda n: next(r['ice'] for r in track_d_rows if r['model']==n))
mu_te, sig_te, y_te = predict_feature_head(track_d_heads[best_d], FEAT_LOADERS['test'])
oracle_sig = SPLITS['test']['true_sigma'].numpy()

idx = np.random.choice(len(sig_te), 400, replace=False)
axes[1].scatter(oracle_sig[idx], sig_te[idx], s=10, alpha=0.5, color='#2C4A6E')
lo, hi = oracle_sig.min(), oracle_sig.max()
axes[1].plot([lo, hi], [lo, hi], 'r--', lw=1, label='oracle=pred')
axes[1].set_xlabel('Oracle σ (ground truth)')
axes[1].set_ylabel('Predicted σ')
axes[1].set_title(f'Track D – predicted vs oracle σ ({best_d})')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/track_d_results.png', dpi=120)
plt.show()
print(f"Best Track D variant: {best_d}")
print("Saved figures/track_d_results.png")

---
## Final Deliverable Table

Aggregate results from all tracks into the table specified in NEXT_STEPS:

> RMSE · MAE · NLL · ICE · coverage@{50,80,90,95}% · bias-by-group · Spearman · top-k risk capture

Success criteria:
- ✅ ICE ≤ 0.02
- ✅ NLL < 1.85 without RMSE degradation > 1%
- ✅ Spearman(σ, |error|) significantly > 0

In [ ]:
df_all = pd.DataFrame(ALL_RESULTS)

# Compute bias-by-group for every model that went through the test set
# using the standard (non-feature-augmented) loaders
mut_type_te = SPLITS['test']['mut_type'].numpy()
mut_label_map = {0: 'synonymous', 1: 'polar', 2: 'nonpolar', 3: 'charge'}

# ── Pretty display ────────────────────────────────────────────────────────────
display_cols = [
    "model", "track", "rmse", "mae", "nll", "ice",
    "cov@0.50", "cov@0.80", "cov@0.90", "cov@0.95",
    "spearman", "top0.10", "top0.20", "top0.30",
]
existing_cols = [c for c in display_cols if c in df_all.columns]

# Keep one row per unique (model, track) combination — drop scaling variants for cleanliness
df_clean = df_all[~df_all['model'].str.contains('scale', case=False, na=False)]
df_clean = df_clean[existing_cols].copy()

# Mark success criteria
if 'ice' in df_clean.columns:
    df_clean['✅ICE≤0.02'] = df_clean['ice'].apply(
        lambda x: '✓' if pd.notna(x) and x <= 0.02 else ''
    )
if 'spearman' in df_clean.columns:
    df_clean['✅Sp>0'] = df_clean['spearman'].apply(
        lambda x: '✓' if pd.notna(x) and x > 0.05 else ''
    )

float_cols = [c for c in df_clean.columns if df_clean[c].dtype == float]
print(f"Full deliverable table  ({len(df_clean)} models)")
print(df_clean.sort_values(['track','ice'], na_position='last')
      .to_string(index=False, float_format="{:.4f}".format))

In [ ]:
# ── Best-per-track summary ────────────────────────────────────────────────────
print("Best model per track (by ICE):")
for track, sub in df_all.groupby('track'):
    sub_p = sub.dropna(subset=['ice'])
    if sub_p.empty: continue
    best = sub_p.loc[sub_p['ice'].idxmin()]
    sp_val = best.get('spearman', float('nan'))
    print(
        f"  Track {track}: {best['model']:50s}  "
        f"ICE={best['ice']:.4f}  NLL={best.get('nll', float('nan')):.4f}  "
        f"RMSE={best['rmse']:.4f}  Spearman={sp_val:.4f}"
    )

print()
# ── Success criteria check ────────────────────────────────────────────────────
prob_rows = df_all.dropna(subset=['ice', 'nll', 'rmse'])
best_ice_row = prob_rows.loc[prob_rows['ice'].idxmin()]
best_nll_row = prob_rows.loc[prob_rows['nll'].idxmin()]
best_sp_row  = prob_rows.loc[prob_rows['spearman'].idxmax()] if 'spearman' in prob_rows else None

print("Success criteria (NEXT_STEPS):")
ice_ok = best_ice_row['ice'] <= 0.02
nll_ok = best_nll_row['nll'] < 1.85
sp_ok  = best_sp_row['spearman'] > 0 if best_sp_row is not None else False

print(f"  ICE ≤ 0.02           : {'✓ PASS' if ice_ok else '✗ FAIL'}  "
      f"(best ICE = {best_ice_row['ice']:.4f} from {best_ice_row['model']})")
print(f"  NLL < 1.85           : {'✓ PASS' if nll_ok else '✗ FAIL'}  "
      f"(best NLL = {best_nll_row['nll']:.4f} from {best_nll_row['model']})")
if best_sp_row is not None:
    print(f"  Spearman(σ,|e|) > 0  : {'✓ PASS' if sp_ok else '✗ FAIL'}  "
          f"(best = {best_sp_row['spearman']:.4f} from {best_sp_row['model']})")

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── (a) NLL vs ICE, all models, coloured by track ─────────────────────────────
ax = fig.add_subplot(gs[0, 0])
track_colors = {'A': '#0D7377', 'B': '#E8913A', 'C': '#2C4A6E', 'D': '#9B5FA7'}
for track, sub in df_all.dropna(subset=['nll','ice']).groupby('track'):
    ax.scatter(sub['nll'], sub['ice'], s=50, alpha=0.8,
               color=track_colors.get(track,'grey'), label=f'Track {track}')
ax.set_xlabel('NLL'); ax.set_ylabel('ICE')
ax.set_title('All models: NLL vs ICE'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ── (b) Spearman bar, best model per track ────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
best_per_track = {}
for track, sub in df_all.dropna(subset=['ice']).groupby('track'):
    best_per_track[track] = sub.loc[sub['ice'].idxmin()]

tracks_ordered = sorted(best_per_track)
sp_vals = [best_per_track[t].get('spearman', 0.0) for t in tracks_ordered]
bars = ax2.bar(tracks_ordered, sp_vals,
               color=[track_colors.get(t,'grey') for t in tracks_ordered], alpha=0.85)
ax2.axhline(0, color='k', lw=0.8)
ax2.set_xlabel('Track'); ax2.set_ylabel('Spearman(σ, |error|)')
ax2.set_title('Best model per track: Spearman'); ax2.grid(axis='y', alpha=0.3)
for bar, v in zip(bars, sp_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.3f}', ha='center', fontsize=9)

# ── (c) Top-k risk capture, best-per-track ────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
k_labels = ['0.10', '0.20', '0.30']
for track in tracks_ordered:
    row = best_per_track[track]
    vals = [row.get(f'top{k}', float('nan')) for k in k_labels]
    ax3.plot(k_labels, vals, 'o-', lw=1.5, ms=6,
             color=track_colors.get(track,'grey'), label=f'Track {track}')
ax3.plot(k_labels, [float(k) for k in k_labels], '--', color='#888', lw=0.8, label='random')
ax3.set_xlabel('k (fraction)'); ax3.set_ylabel('Top-k risk capture')
ax3.set_title('Top-k risk capture (best per track)'); ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

# ── (d) ICE bar, best per track ───────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
ice_vals = [best_per_track[t]['ice'] for t in tracks_ordered]
bars2 = ax4.bar(tracks_ordered, ice_vals,
                color=[track_colors.get(t,'grey') for t in tracks_ordered], alpha=0.85)
ax4.axhline(0.02, color='red', linestyle='--', lw=1.2, label='target ≤0.02')
ax4.set_xlabel('Track'); ax4.set_ylabel('ICE')
ax4.set_title('Best ICE per track'); ax4.legend(fontsize=8); ax4.grid(axis='y', alpha=0.3)
for bar, v in zip(bars2, ice_vals):
    ax4.text(bar.get_x() + bar.get_width()/2, v + 0.001, f'{v:.4f}', ha='center', fontsize=9)

# ── (e) Reliability diagram, best per track ───────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot([0,1],[0,1],'--',color='#888',lw=0.8,label='perfect')

# Map tracks to their best heads
def get_test_predictions_for_track(track):
    row = best_per_track.get(track)
    if row is None: return None
    name = row['model']
    # Find in one of our head dicts
    for d in [track_a_heads, track_b_heads_A, track_c_heads, track_d_heads]:
        head = d.get(name)
        if head is not None:
            if track == 'D':
                return predict_feature_head(head, FEAT_LOADERS['test'])
            return predict(head, LOADERS['test'])
    # Try stripping scaling suffix
    base = name.split('[')[0].strip()
    for d in [track_a_heads, track_b_heads_A, track_c_heads]:
        head = d.get(base)
        if head is not None:
            return predict(head, LOADERS['test'])
    return None

for track in tracks_ordered:
    preds = get_test_predictions_for_track(track)
    if preds is None: continue
    mu, sig, y = preds
    if sig is None: continue
    nom, emp = compute_coverage_curve(mu, sig, y)
    ax5.plot(nom, emp, 'o-', ms=3, lw=1.4,
             color=track_colors.get(track,'grey'), label=f'Track {track}')
ax5.set_xlim(0,1); ax5.set_ylim(0,1); ax5.set_aspect('equal')
ax5.set_xlabel('Nominal'); ax5.set_ylabel('Empirical')
ax5.set_title('Reliability diagram (best per track)'); ax5.legend(fontsize=8); ax5.grid(alpha=0.3)

# ── (f) RMSE comparison, all tracks ─────────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
for track in tracks_ordered:
    sub = df_all[df_all['track']==track].dropna(subset=['rmse'])
    ax6.scatter([track]*len(sub), sub['rmse'], s=30, alpha=0.6,
                color=track_colors.get(track,'grey'))
ax6.set_xlabel('Track'); ax6.set_ylabel('RMSE')
ax6.set_title('RMSE distribution per track'); ax6.grid(axis='y', alpha=0.3)

plt.savefig('figures/deliverable_summary.png', dpi=130)
plt.show()
print("Saved figures/deliverable_summary.png")

In [ ]:
# ── Save deliverable table as CSV ─────────────────────────────────────────────
out_csv = Path('figures/next_steps_deliverable.csv')
df_all.to_csv(out_csv, index=False, float_format='%.5f')
print(f"Deliverable table saved to {out_csv}")
print(f"Total models: {len(df_all)}")
print()

# ── Compact mentor-facing summary ─────────────────────────────────────────────
print("=" * 78)
print("SUMMARY FOR MENTOR MEETING")
print("=" * 78)
print(f"{'Track':<6} {'Best model':<45} {'ICE':>7} {'NLL':>7} {'RMSE':>7} {'Spearman':>10}")
print("-" * 78)
for track in tracks_ordered:
    row = best_per_track[track]
    name = row['model']
    if len(name) > 44: name = name[:41] + '...'
    sp = row.get('spearman', float('nan'))
    print(
        f"{track:<6} {name:<45} "
        f"{row['ice']:7.4f} {row.get('nll', float('nan')):7.4f} "
        f"{row['rmse']:7.4f} {sp:10.4f}"
    )
print("=" * 78)